# Cuaderno 2: Simulación del tsunami en TPU

Este cuaderno ejecuta la simulación de propagación del tsunami del 12 de diciembre
de 1979 (Mw 8.2, Tumaco, Colombia) utilizando el modelo **tsunamiTPUlab v1.0**
(Smarras et al., 2023, *Geoscientific Model Development*).

**⚠ REQUISITO:** Activar runtime **TPU** antes de ejecutar.
En Google Colab: `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → `TPU`.

**Tiempo estimado:** ~5–10 minutos en TPUv2 (8 cores).

**Prerrequisito:** Ejecutar primero `01-bathymetry.ipynb` para generar `tumaco_dem_utm_sim.tif`.

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # TF 2.12 es la última versión con soporte completo de la API TPU usada por tsunamiTPUlab
    !pip install -q "tensorflow==2.12.0" scikit-image rasterio pyproj
    import os
    if not os.path.exists('/content/tsunamiTPUlab'):
        !git clone --quiet https://github.com/smarras79/tsunamiTPUlab.git /content/tsunamiTPUlab
    sys.path.insert(0, '/content/tsunamiTPUlab')
    print("✓ Dependencias listas")
else:
    print("Modo local – la simulación TPU solo funciona en Google Colab con runtime TPU.")

In [ ]:
import os
import numpy as np
import requests
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import gaussian_filter

import rasterio
from rasterio.crs import CRS
from rasterio.transform import from_bounds
from rasterio.warp import calculate_default_transform, reproject, Resampling

WORK_DIR   = Path('/content') if IN_COLAB else Path('.')
MODEL_DIR  = Path('/content/tsunamiTPUlab') if IN_COLAB else Path('./tsunamiTPUlab')
OUTPUT_DIR = WORK_DIR / 'output_1979'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEM_PATH = WORK_DIR / 'tumaco_dem_utm_sim.tif'

# ── Regenerar DEM si no existe (sesión nueva en Colab) ──────────────────────
if not DEM_PATH.exists():
    print("DEM no encontrado – generando desde cero...")

    LAT_MIN, LAT_MAX =  0.5,  3.5
    LON_MIN, LON_MAX = -79.5, -77.0
    CRS_SRC  = CRS.from_epsg(4326)
    CRS_DEST = CRS.from_epsg(32618)

    geo_path  = WORK_DIR / 'tumaco_bathy_geo.tif'
    etopo_path = WORK_DIR / 'tumaco_bathy_etopo.tif'

    # Intento 1: GEBCO 2023
    bathy_ok = False
    try:
        url = (f"https://download.gebco.net/a/gebco_2023"
               f"?lat1={LAT_MIN}&lat2={LAT_MAX}&lon1={LON_MIN}&lon2={LON_MAX}&format=netcdf4")
        r = requests.get(url, timeout=120)
        if r.status_code == 200 and len(r.content) > 5000:
            import xarray as xr, io
            ds = xr.open_dataset(io.BytesIO(r.content))
            varname = [v for v in ds.data_vars if 'elev' in v.lower() or 'z' in v.lower()][0]
            da = ds[varname]
            lat_dim = [d for d in da.dims if 'lat' in d.lower()][0]
            lon_dim = [d for d in da.dims if 'lon' in d.lower()][0]
            if da[lat_dim].values[0] < da[lat_dim].values[-1]:
                da = da.isel({lat_dim: slice(None, None, -1)})
            arr = da.values.astype('float32')
            lats, lons = da[lat_dim].values, da[lon_dim].values
            tf_geo = from_bounds(lons.min(), lats.min(), lons.max(), lats.max(),
                                 arr.shape[1], arr.shape[0])
            with rasterio.open(geo_path, 'w', driver='GTiff',
                               height=arr.shape[0], width=arr.shape[1],
                               count=1, dtype='float32', crs=CRS_SRC,
                               transform=tf_geo) as dst:
                dst.write(arr, 1)
            bathy_ok = True
            print("✓ GEBCO 2023 descargado.")
    except Exception as e:
        print(f"✗ GEBCO: {e}")

    # Intento 2: ETOPO 2022 (NOAA REST)
    if not bathy_ok:
        try:
            ncols = int((LON_MAX - LON_MIN) / 0.004167)
            nrows = int((LAT_MAX - LAT_MIN) / 0.004167)
            url2 = (
                "https://gis.ngdc.noaa.gov/arcgis/rest/services/DEM_mosaics/"
                "DEM_global_mosaic/ImageServer/exportImage"
                f"?bbox={LON_MIN},{LAT_MIN},{LON_MAX},{LAT_MAX}"
                f"&bboxSR=4326&size={ncols},{nrows}&imageSR=4326"
                "&format=tiff&pixelType=F32&f=image"
            )
            r2 = requests.get(url2, timeout=120)
            r2.raise_for_status()
            with open(geo_path, 'wb') as f:
                f.write(r2.content)
            bathy_ok = True
            print("✓ ETOPO 2022 descargado.")
        except Exception as e:
            print(f"✗ ETOPO: {e}")

    # Intento 3: DEM sintético
    if not bathy_ok:
        print("⚠ Usando DEM sintético (solo para pruebas).")
        res = 0.004167
        lons_s = np.arange(LON_MIN, LON_MAX, res)
        lats_s = np.arange(LAT_MIN, LAT_MAX, res)
        nr, nc = len(lats_s), len(lons_s)
        x_rel = (lons_s - LON_MIN) / (LON_MAX - LON_MIN)
        z = np.where(x_rel < 0.70, -4000 + 3800*(x_rel/0.70),
                     np.where(x_rel < 0.88, -200 + 200*((x_rel-0.70)/0.18), 10.0))
        dem_s = np.tile(z, (nr, 1)).astype('float32')
        dem_s += gaussian_filter(np.random.randn(nr, nc) * 50, sigma=5)
        tf_s = from_bounds(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX, nc, nr)
        with rasterio.open(geo_path, 'w', driver='GTiff', height=nr, width=nc,
                           count=1, dtype='float32', crs=CRS_SRC, transform=tf_s) as dst:
            dst.write(dem_s, 1)

    # Reproyectar a UTM 18N
    utm_path = WORK_DIR / 'tumaco_dem_utm.tif'
    with rasterio.open(geo_path) as src:
        tf_utm, w_utm, h_utm = calculate_default_transform(
            src.crs, CRS_DEST, src.width, src.height, *src.bounds)
        meta = src.meta.copy()
        meta.update({'crs': CRS_DEST, 'transform': tf_utm,
                     'width': w_utm, 'height': h_utm, 'dtype': 'float32'})
        with rasterio.open(utm_path, 'w', **meta) as dst:
            reproject(source=rasterio.band(src, 1),
                      destination=rasterio.band(dst, 1),
                      src_transform=src.transform, src_crs=src.crs,
                      dst_transform=tf_utm, dst_crs=CRS_DEST,
                      resampling=Resampling.bilinear)

    # Invertir eje X (océano al este → océano al oeste en coordenada y del modelo)
    with rasterio.open(utm_path) as src:
        arr_utm = src.read(1)
        old_tf  = src.transform
        meta2   = src.meta.copy()
    arr_flip = arr_utm[:, ::-1]
    new_tf = rasterio.transform.from_origin(
        west=old_tf.c + old_tf.a * arr_utm.shape[1],
        north=old_tf.f,
        xsize=-old_tf.a,
        ysize=-old_tf.e
    )
    meta2['transform'] = new_tf
    with rasterio.open(DEM_PATH, 'w', **meta2) as dst:
        dst.write(arr_flip, 1)
    print(f"✓ DEM listo: {DEM_PATH}")

# Verificar
with rasterio.open(str(DEM_PATH)) as src:
    _shape = (src.height, src.width)
    _res   = abs(src.transform.a)
print(f"DEM de entrada: {DEM_PATH.name} | {_shape[0]}×{_shape[1]} px | {_res:.0f} m/px")
print(f"Directorio de salida: {OUTPUT_DIR}")

## 1. Configuración de la sesión TPU

Este bloque inicializa la conexión con las TPUs disponibles en Colab.
Requiere que el runtime esté configurado como **TPU**.

In [ ]:
# Configuración para uso en Colab público con TPU
PUBLIC_COLAB = IN_COLAB
BUCKET = ''           # No se usa GCS – guardamos localmente en /content/
PROJECT_ID = ''       # No se usa GCS en este flujo simplificado
TPU_WORKER = ''

if PUBLIC_COLAB:
    # Verificar disponibilidad de TPU
    try:
        import tensorflow.compat.v1 as tf
        tf.disable_v2_behavior()
        TPU_WORKER = 'grpc://' + os.environ.get('COLAB_TPU_ADDR', '')
        if not os.environ.get('COLAB_TPU_ADDR'):
            raise RuntimeError("Variable COLAB_TPU_ADDR no encontrada.")
        print(f"✓ TPU detectada: {TPU_WORKER}")
    except Exception as e:
        raise RuntimeError(
            f"TPU no disponible: {e}\n"
            "Active el runtime TPU: Entorno de ejecución → Cambiar tipo → TPU"
        )

## 2. Cargar el modelo tsunamiTPUlab

In [ ]:
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

# Cargar las utilidades del modelo
os.chdir(str(MODEL_DIR))  # necesario para importar user_constants y tpu_simulation_utilities

# Crear user_constants.py temporal con la configuración requerida
user_constants_content = f"""
BUCKET = ''
PROJECT_ID = ''
PUBLIC_COLAB = {PUBLIC_COLAB}
TPU_WORKER = '{TPU_WORKER}'
"""
with open(MODEL_DIR / 'user_constants.py', 'w') as f:
    f.write(user_constants_content)

import tpu_simulation_utilities as tsu
from typing import Optional, Sequence, Text, Callable
import rasterio

print("✓ tsunamiTPUlab cargado.")

## 3. Parámetros de la simulación

Los parámetros están calibrados para el evento del 12 de diciembre de 1979 (Mw 8.2).

**Condición CFL:** $dt \leq dx / (\sqrt{2} \cdot c_{max})$ donde $c_{max} = \sqrt{gH_{max}}$
- Para $H_{max} \approx 5000$ m: $c_{max} \approx 221$ m/s
- Para $dx \approx 450$ m: $dt \leq 1.43$ s → usamos $dt = 1.0$ s

In [ ]:
# =========================================================
# PARÁMETROS DE SIMULACIÓN – ajustar si se cambia el DEM
# =========================================================

# Obtener resolución real del DEM
with rasterio.open(str(DEM_PATH)) as src:
    RESOLUTION = int(abs(src.transform.a))  # metros/píxel
    DEM_SHAPE  = (src.height, src.width)

# Paso de tiempo (condición CFL)
DT = 1.0          # segundos (CFL seguro para 450m/pixel y profundidad ~5000m)

# Cores TPU: 1×8 para Colab TPUv2 (8 cores)
CX = 1            # cores en dirección x (filas)
CY = 8            # cores en dirección y (columnas)

# Duración de la simulación
START_TIME     = 0       # segundos
NUM_SECS       = 1800.0  # 30 minutos (cubre el primer arribo + 15 min post-arribo)
SECS_PER_CYCLE = 60.0    # guardar snapshot cada 60 s

# Directorio de salida (local en Colab – no requiere GCS)
RUN_DIR = str(OUTPUT_DIR)

print("PARÁMETROS DE SIMULACIÓN")
print("-" * 40)
print(f"  DEM:        {DEM_PATH.name}")
print(f"  Shape:      {DEM_SHAPE[0]} filas × {DEM_SHAPE[1]} columnas")
print(f"  Resolución: {RESOLUTION} m/pixel")
print(f"  dt:         {DT} s")
print(f"  Duración:   {NUM_SECS:.0f} s ({NUM_SECS/60:.0f} min)")
print(f"  Ciclos:     {NUM_SECS/SECS_PER_CYCLE:.0f} snapshots")
print(f"  TPU cores:  {CX}×{CY} = {CX*CY}")
print(f"  Salida:     {RUN_DIR}")

## 4. Condición inicial: N-wave para el evento de 1979

La N-wave bidimensional de Carrier {cite:t}`carrier2003` aproxima la deformación
de la superficie del agua generada por el terremoto de 1979 (Mw 8.2).

**Parametrización física:**

| Parámetro | Valor | Justificación |
|-----------|-------|---------------|
| Amplitud $A$ | 1.5 m | Deformación cosísmica para Mw 8.2 |
| Longitud de onda $\lambda$ | 80 000 m | Típica para ruptura ~250 km |
| Distancia epicentro-costa | ~60 000 m | Herd et al. (1981) |
| Tipo | Leading depression N-wave | Observado en el arribo histórico |

**Orientación del DEM:**
Con el DEM invertido (Cuaderno 1): columna 0 = costa Este, columna N = océano Oeste.
El epicentro está a ~60 km de la costa, es decir en $y \approx L - 60\,000$ m.

In [ ]:
def nwave_tumaco_1979(y, x, Ly, nx, ny, dem):
    """
    Condición inicial de superficie del agua para el tsunami del 12-12-1979.

    Implementa una N-wave de doble Gaussiana tipo Carrier (2003):
      η(y) = 2 * [A1 * exp(-k1*(y-y1)²) - A2 * exp(-k2*(y-y2)²)]

    El DEM está orientado con:
      y=0  → costa de Tumaco (este)
      y=Ly → océano Pacífico abierto (oeste)

    Parámetros calibrados para reproducir run-up observado de 3–5 m
    en Tumaco (Herd et al., 1981).
    """
    # Amplitud y longitud de onda
    A    = 1.5       # metros – amplitud de la elevación principal
    lamb = 80_000.0  # metros – longitud de onda característica

    # Constantes de la N-wave de Carrier (2003)
    k1 = 28.416 / lamb**2   # constante de la elevación principal
    k2 = 256.0  / lamb**2   # constante de la depresión frontal

    # Posición del epicentro en coordenadas del DEM
    # El epicentro está a ~60 km de la costa (y=0), en y ≈ Ly - 60000
    # La ola viaja hacia y decreciente (hacia la costa)
    d_ocean = 60_000.0  # metros desde la costa hasta el epicentro
    y_epi   = Ly - d_ocean  # posición en coordenadas del DEM

    # Pico de elevación (coda de la ola, más oceánica)
    y1 = y_epi + 0.3153 * lamb   # ~25 km detrás del epicentro (más al oeste)
    # Depresión frontal (llega primero a la costa)
    y2 = y_epi                   # en el epicentro (leading depression)

    # N-wave: depresión frontal + elevación posterior
    z = 2.0 * (
        A * tf.math.exp(-k1 * tf.math.square(y - y1))
        - (A / 3.0) * tf.math.exp(-k2 * tf.math.square(y - y2))
    )

    # Extender a grilla 2D (uniforme en x)
    z = tf.expand_dims(z, axis=0)
    z = tf.repeat(z, nx, axis=0)

    # Compensar la altura del fondo (agua sobre batimetría negativa)
    z = z - tf.math.minimum(dem, tf.zeros_like(dem))

    # Aplicar mínimo de profundidad para estabilidad numérica
    z = tf.where(
        z < tsu.SAINT_VENANT_EPS,
        tf.ones_like(z) * tsu.SAINT_VENANT_EPS,
        z
    )
    return z


# Visualización de la N-wave unidimensional
fig, ax = plt.subplots(figsize=(12, 4))
A, lamb = 1.5, 80_000.0
k1, k2 = 28.416 / lamb**2, 256.0 / lamb**2
Ly_approx = RESOLUTION * DEM_SHAPE[1]  # longitud aproximada del dominio en y (metros)
d_ocean = 60_000.0
y_epi = Ly_approx - d_ocean
y1 = y_epi + 0.3153 * lamb
y2 = y_epi

y_vec = np.linspace(0, Ly_approx, 1000)
eta = 2 * (A * np.exp(-k1 * (y_vec - y1)**2) - (A/3) * np.exp(-k2 * (y_vec - y2)**2))

ax.fill_between(y_vec / 1000, eta, 0, where=eta > 0, color='coral', alpha=0.6, label='Elevación')
ax.fill_between(y_vec / 1000, eta, 0, where=eta < 0, color='steelblue', alpha=0.6, label='Depresión')
ax.plot(y_vec / 1000, eta, 'k-', linewidth=1.5)
ax.axvline(0, color='saddlebrown', linestyle='--', linewidth=1, label='Costa (y=0)')
ax.axvline(Ly_approx / 1000, color='navy', linestyle='--', linewidth=1, label='Borde oceánico')
ax.axvline(y_epi / 1000, color='red', linestyle=':', linewidth=1.5, label='Epicentro 1979')
ax.set_xlabel('Distancia desde la costa (km) →')
ax.set_ylabel('Elevación inicial de la superficie (m)')
ax.set_title('Condición inicial N-wave – Evento Tumaco 1979 (Mw 8.2)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, Ly_approx / 1000)
plt.tight_layout()
plt.savefig(WORK_DIR / 'nwave_initial.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Longitud del dominio y: {Ly_approx/1000:.0f} km")
print(f"Epicentro en y = {y_epi/1000:.0f} km desde la costa")

## 5. Definición del gestor de la simulación

Copiado y adaptado de `crescent_city_scenario.ipynb` en tsunamiTPUlab.
Se mantiene el `TPUSimulationManager` y `run_simulation` con condiciones
de contorno abiertas (Neumann) en todos los bordes del dominio.

In [ ]:
class TPUSimulationManager:
    """Gestor de la simulación en TPU."""

    def __init__(self, params, simulation_builder, session_runner_builder):
        self._params = params
        config = tf.ConfigProto(isolate_session_state=True)
        self._sess = tf.Session(config=config, target=TPU_WORKER)
        self._sim = simulation_builder(self._sess)
        self._session_runner = session_runner_builder(self._sim)
        self._sess.run(tf.global_variables_initializer())

    def run_simulation(self, start_time_secs=0.0):
        start_step = int(round(start_time_secs / self._params.dt))
        self._session_runner(self._sess, start_step)

    def __del__(self):
        try:
            tf.tpu.shutdown_system()
        except Exception:
            pass


def get_sim_builder(params, unpadded_dem, unpadded_manning_matrix,
                    h_bcs, q_x_bcs, q_y_bcs, init_files_manager,
                    input_file_format=None, start_time_secs=0.0,
                    water_ic_fn=tsu.constant_height_zeros,
                    flux_ic_fn=tsu.constant_height_zeros):
    init_fn_builder = tsu.SaintVenantRealisticInitFnBuilder(
        params, tsu.INIT_STATE_KEYS, unpadded_dem, unpadded_manning_matrix,
        init_files_manager, input_file_format, start_time_secs,
        water_ic_fn, flux_ic_fn
    )
    step_fn = tsu.SaintVenantRiverChannelStep(
        tsu.ApplyKernelOp(), params, h_bcs, q_x_bcs, q_y_bcs
    )
    step_fn = tsu.DynamicStepUpdater(
        step_fn,
        num_secs_per_while_loop=params.num_secs_per_cycle,
        dt=params.dt
    )
    step_fn = tsu.build_step_fn(
        params, step_fn, tsu.STATE_KEYS, tsu.ADDITIONAL_STATE_KEYS
    )

    def sim_builder(sess):
        topology = tf.tpu.experimental.Topology(
            sess.run(tf.tpu.initialize_system())
        )
        return tsu.TPUSimulation(
            init_fn=init_fn_builder.init_fn,
            step_fn=step_fn,
            computation_shape=params.computation_shape,
            tpu_topology=topology,
            host_vars_filter=tsu.INIT_STATE_KEYS
        )
    return sim_builder


def run_simulation_tumaco(dem_path, resolution, num_secs, num_secs_per_cycle,
                          dt, cx, cy, run_dir, start_time_secs=0.0):
    """
    Ejecuta la simulación de propagación del tsunami de 1979 en Tumaco.

    Adaptado de crescent_city_scenario.ipynb (tsunamiTPUlab v1.0).
    Condiciones de contorno: Neumann abiertas en todos los bordes.
    Condición inicial: N-wave tipo Carrier para evento Mw 8.2.
    """
    slope = 1e-4  # pendiente de referencia para condiciones de contorno
    input_file_format = None  # sin reinicio desde checkpoint

    # Cargar DEM
    unpadded_dem, geoinfo = tsu.load_dem_from_tiff_file(
        str(dem_path), resolution, smooth_kernel=5
    )
    print(f"DEM cargado: {unpadded_dem.shape} | "
          f"min={unpadded_dem.min():.0f}m max={unpadded_dem.max():.0f}m")

    # Matriz de Manning uniforme
    unpadded_river_mask = np.ones_like(unpadded_dem, dtype=bool)
    unpadded_manning_matrix = tsu.get_manning_matrix_from_river_mask(
        unpadded_river_mask,
        tsu.MANNING_COEFF_RIVER,
        tsu.MANNING_COEFF_FLOODPLAIN
    )

    # Parámetros de la grilla
    params = tsu.get_sv_params(
        unpadded_dem.shape, resolution, num_secs, num_secs_per_cycle, cx, cy, dt
    )
    print(f"Params: nx={params.nx}, ny={params.ny}, "
          f"steps={params.num_steps}, cycles={params.num_cycles}")

    # ---- Condiciones de contorno ----
    # Todos los bordes: Neumann (flujo libre, condición absorvente)
    bc_kwargs = dict(
        fraction_start=0, fraction_end=1.0,
        left_padding=params.left_padding,
        top_padding=params.top_padding,
        slope=slope,
        value=0.0,
        unpadded_dem=unpadded_dem,
        unpadded_manning_matrix=unpadded_manning_matrix
    )

    h_bcs = [
        tsu.NeumannBoundary(boundary_side=tsu.BoundarySide.LEFT, **bc_kwargs),
        tsu.NeumannBoundary(boundary_side=tsu.BoundarySide.RIGHT, **bc_kwargs),
    ]
    q_y_bcs = [
        tsu.NeumannBoundary(boundary_side=tsu.BoundarySide.LEFT, **bc_kwargs),
        tsu.DirichletBoundary(boundary_side=tsu.BoundarySide.RIGHT, **bc_kwargs),
    ]
    q_x_bcs = [
        tsu.NeumannBoundary(boundary_side=tsu.BoundarySide.TOP, **bc_kwargs),
        tsu.NeumannBoundary(boundary_side=tsu.BoundarySide.BOTTOM, **bc_kwargs),
    ]

    # Gestor de archivos de inicialización
    init_files_manager = tsu.InitFilesManager(
        params, tsu.three_d_subgrid_of_2d_grid, run_dir
    )

    # Constructor de simulación con N-wave de 1979
    sim_builder = get_sim_builder(
        params, unpadded_dem, unpadded_manning_matrix,
        h_bcs, q_x_bcs, q_y_bcs, init_files_manager,
        input_file_format, start_time_secs,
        water_ic_fn=nwave_tumaco_1979,
        flux_ic_fn=tsu.constant_height_zeros
    )

    def session_runner_builder(sim):
        return tsu.get_session_runner_builder(
            params, init_files_manager, run_dir, sim
        )

    # Ejecutar
    manager = TPUSimulationManager(params, sim_builder, session_runner_builder)
    manager.run_simulation(start_time_secs)
    print("✓ Simulación completada.")


print("Funciones de simulación definidas.")

## 6. Ejecutar la simulación

**⚠ Esta celda solo funciona con runtime TPU activo en Google Colab.**
Duración estimada: 5–10 minutos.

In [ ]:
import time

if PUBLIC_COLAB:
    print("Iniciando simulación TPU...")
    print(f"  DEM:      {DEM_PATH.name}")
    print(f"  Duración: {NUM_SECS:.0f} s ({NUM_SECS/60:.0f} min)")
    print(f"  Salida:   {RUN_DIR}")
    print()

    t0 = time.time()
    g = tf.Graph()
    with g.as_default():
        run_simulation_tumaco(
            dem_path=DEM_PATH,
            resolution=RESOLUTION,
            num_secs=NUM_SECS,
            num_secs_per_cycle=SECS_PER_CYCLE,
            dt=DT,
            cx=CX,
            cy=CY,
            run_dir=RUN_DIR,
            start_time_secs=START_TIME
        )

    elapsed = time.time() - t0
    print(f"\nTiempo de cómputo: {elapsed/60:.1f} minutos")
else:
    print("Modo local – omitiendo ejecución TPU.")
    print("Para ejecutar: abrir este notebook en Google Colab con runtime TPU.")

## 7. Verificar archivos de salida

In [ ]:
import glob

output_files = sorted(glob.glob(f'{RUN_DIR}/h-*.np'))

if output_files:
    print(f"✓ Archivos de salida encontrados: {len(output_files)}")
    print(f"  Primero: {Path(output_files[0]).name}")
    print(f"  Último:  {Path(output_files[-1]).name}")
    total_mb = sum(Path(f).stat().st_size for f in output_files) / 1e6
    print(f"  Tamaño total: {total_mb:.1f} MB")
else:
    print("⚠ No se encontraron archivos de salida.")
    print("  Si ejecutó la simulación correctamente, revise el directorio:")
    print(f"  {RUN_DIR}")

# Listar todos los archivos en el directorio de salida
print("\nContenido del directorio de salida:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}")

## 8. Vista previa de un snapshot

Verificación rápida de los resultados cargando el snapshot del minuto 15
(tiempo estimado de llegada del tsunami a Tumaco según registros históricos).

In [ ]:
import rasterio

# Cargar el DEM para referencia
with rasterio.open(str(DEM_PATH)) as src:
    dem_sim = src.read(1)

# Buscar el snapshot en t=900s (15 minutos = tiempo estimado de arribo)
t_target = 900.0
t_files = {float(Path(f).stem.split('-')[1]): f for f in output_files}

# Buscar el snapshot más cercano a t=900s
if t_files:
    t_closest = min(t_files.keys(), key=lambda t: abs(t - t_target))
    hmap = np.load(t_files[t_closest])

    # Calcular inundación sobre tierra
    inundation = np.where(
        dem_sim >= 0,
        hmap + dem_sim,   # profundidad de agua sobre tierra
        0.0
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    im1 = axes[0].imshow(hmap, cmap='RdBu_r', vmin=-3, vmax=3)
    plt.colorbar(im1, ax=axes[0], label='Altura del agua (m)')
    axes[0].set_title(f'Superficie del agua — t = {t_closest:.0f} s ({t_closest/60:.0f} min)')

    im2 = axes[1].imshow(inundation, cmap='YlOrRd', vmin=0, vmax=5)
    plt.colorbar(im2, ax=axes[1], label='Inundación en tierra (m)')
    axes[1].set_title('Inundación sobre nivel del terreno')

    plt.suptitle('Simulación Tsunami Tumaco 1979 — Vista previa', fontsize=12)
    plt.tight_layout()
    plt.savefig(WORK_DIR / 'preview_t900s.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Inundación máxima a t={t_closest:.0f}s: {inundation.max():.2f} m")
else:
    print("Sin archivos de salida para visualizar.")